# 🧪 Python 数据清洗基础 — 笔试/面试专项练习

**规则**: 每道题有一个空 Cell 供你作答，紧跟一个折叠的参考答案 Cell。
先自己写，写完再对答案！

---

In [1]:
import pandas as pd
import numpy as np
import re
print('✅ Ready')

✅ Ready


---
## Part 1: 字符串 & 正则 (String / Regex)
---

### Q1. re.sub vs str.replace
给定字符串 `text = "Price: $49.99! (Sale)"`, 用**两种方式**去掉所有非字母非数字字符，只保留字母、数字和空格。
- 方式 A: 使用 `re.sub()`
- 方式 B: 使用 Pandas `.str.replace()` (先转成 Series)

In [7]:
text = "Price: $49.99! (Sale)"
# 你的答案
result_a = re.sub(r'[^a-zA-Z0-9\s]','',text)
s = pd.Series(text)
result_b = s.replace(r'[^a-zA-Z0-9\s]','',regex=True)

print(f'A:{result_a}')
print(F'B:{result_b.values[0]}')



A:Price 4999 Sale
B:Price 4999 Sale


In [6]:
# ✅ Q1 参考答案
text = "Price: $49.99! (Sale)"

# 方式 A: re.sub — 参数顺序: (pattern, replacement, string)
result_a = re.sub(r'[^a-zA-Z0-9\s]', '', text)
print(f'A: {result_a}')  # "Price 4999 Sale"

# 方式 B: Pandas .str.replace — 注意 regex=True
s = pd.Series([text])
result_b = s.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
print(f'B: {result_b.values[0]}')

# 💡 关键区别:
# re.sub()         → 用于单条字符串, 写在 apply 里
# .str.replace()   → 用于整列 Series, 直接向量化, 不需要 apply

A: Price 4999 Sale
B: Price 4999 Sale


### Q2. 原生 replace vs re.sub
给定 `text = "hello-world_foo@bar.com"`, 请问以下两行代码的输出分别是什么？（先想，再运行验证）
```python
print(text.replace('-', ' '))
print(re.sub(r'[^a-zA-Z]', ' ', text))
```

In [8]:
text = "hello-world_foo@bar.com"
# 先在脑子里想输出，再运行验证
#a:hello world_foo@bar.com
#b:hello world foo bar com
print(text.replace('-',' '))
print(re.sub(r'[^a-zA-Z]',' ',text))

hello world_foo@bar.com
hello world foo bar com


In [ ]:
# ✅ Q2 参考答案
text = "hello-world_foo@bar.com"

# .replace() — 只替换一个字符, 其他不动
print(text.replace('-', ' '))   # "hello world_foo@bar.com"

# re.sub() — 正则匹配所有非字母字符, 全部替换
print(re.sub(r'[^a-zA-Z]', ' ', text))  # "hello world foo bar com"

# 💡 str.replace() 不支持正则! 只能做简单的一对一替换
# 需要批量替换时, 必须用 re.sub()

### Q3. apply + re.sub 的正确写法
给定 DataFrame，将 `review` 列中的所有数字替换为 `<NUM>`。
```python
df = pd.DataFrame({'review': ['I bought 3 items for $29.99', 'Rating: 5 stars', None]})
```
注意处理 NaN！

In [16]:
df = pd.DataFrame({'review': ['I bought 3 items for $29.99', 'Rating: 5 stars', None]})
# 你的答案
pd.Series(df['review']).replace(r'\d+\.?\d*','<NUM>',regex=True)



0    I bought <NUM> items for $<NUM>
1                Rating: <NUM> stars
2                                NaN
Name: review, dtype: str

In [11]:
# ✅ Q3 参考答案
df = pd.DataFrame({'review': ['I bought 3 items for $29.99', 'Rating: 5 stars', None]})

# 方法 1: apply + re.sub (需要手动处理 NaN)
df['clean_v1'] = df['review'].apply(lambda x: re.sub(r'\d+', '<NUM>', x) if pd.notna(x) else x)

# 方法 2: .str.replace (自动跳过 NaN, 更推荐!)
df['clean_v2'] = df['review'].str.replace(r'\d+', '<NUM>', regex=True)

print(df[['review', 'clean_v1', 'clean_v2']])

# 💡 .str.replace 天然兼容 NaN, apply 需要手动判断 pd.notna()

                        review                               clean_v1  \
0  I bought 3 items for $29.99  I bought <NUM> items for $<NUM>.<NUM>   
1              Rating: 5 stars                    Rating: <NUM> stars   
2                          NaN                                    NaN   

                                clean_v2  
0  I bought <NUM> items for $<NUM>.<NUM>  
1                    Rating: <NUM> stars  
2                                    NaN  


---
## Part 2: 排序 & 聚合链 (Sort / Agg Chain)
---

### Q4. sort_values vs value_counts 的覆盖陷阱
给定以下数据，请按**评分从低到高**输出每个评分的占比和累积占比。
```python
scores = pd.Series([5,5,5,4,4,3,2,1,1,5])
```
要求输出格式：评分 1→2→3→4→5 依次排列。

In [18]:
scores = pd.Series([5,5,5,4,4,3,2,1,1,5])
# 你的答案
scores.sort_index(ascending=False)

9    5
8    1
7    1
6    2
5    3
4    4
3    4
2    5
1    5
0    5
dtype: int64

In [ ]:
# ✅ Q4 参考答案
scores = pd.Series([5,5,5,4,4,3,2,1,1,5])

# ❌ 错误写法 — sort_values 排的是原数据行, 被 value_counts 覆盖
# scores.sort_values().value_counts()  # 还是按频次排!

# ✅ 正确 — 先 value_counts, 再 sort_index
result = scores.value_counts(normalize=True).sort_index()
result_cum = result.cumsum()

print('占比:')
print(result)
print('\n累积占比:')
print(result_cum)

# 💡 value_counts() 永远按频次排, sort_index() 按标签排




### Q5. groupby + agg 多指标
给定以下订单数据，请按 `category` 分组，同时计算：订单数、平均金额、最大金额。按平均金额降序排列。
```python
orders = pd.DataFrame({
    'category': ['电子','服饰','电子','食品','服饰','食品','电子'],
    'amount': [299, 89, 1599, 45, 168, 32, 899]
})
```

In [24]:
orders = pd.DataFrame({
    'category': ['电子','服饰','电子','食品','服饰','食品','电子'],
    'amount': [299, 89, 1599, 45, 168, 32, 899]
})
# 你的答案
orders.groupby('category').agg(
    total_orders_cnt = ('amount','count'),
    avg_amount = ('amount','sum'),
    max_amount = ('amount','max')

).sort_values('avg_amount',ascending=False)





,total_orders_cnt,avg_amount,max_amount
category,,,
电子,3,2797,1599
服饰,2,257,168
食品,2,77,45


In [25]:
# ✅ Q5 参考答案
orders = pd.DataFrame({
    'category': ['电子','服饰','电子','食品','服饰','食品','电子'],
    'amount': [299, 89, 1599, 45, 168, 32, 899]
})

result = orders.groupby('category')['amount'].agg(
    order_count='count',
    avg_amount='mean',
    max_amount='max'
).sort_values('avg_amount', ascending=False)

print(result)

# 💡 agg() 里用 命名聚合 (named agg) 可以同时算多指标并起别名

          order_count  avg_amount  max_amount
category                                     
电子                  3  932.333333        1599
服饰                  2  128.500000         168
食品                  2   38.500000          45


### Q6. 赋值陷阱
以下代码有什么问题？运行后 `df` 是否真的被修改了？
```python
df = pd.DataFrame({'price': ['$10', '$25', '$8']})
df['price'].str.replace('$', '')
df['price'].astype(float)
```

In [29]:
df = pd.DataFrame({'price': ['$10', '$25', '$8']})
# 先想: 运行后 df['price'] 是什么类型? 值是什么?
# 然后写出正确版本
df['price'] = df['price'].str.replace('$','').astype(float)
df['price'].dtype

dtype('float64')

In [ ]:
# ✅ Q6 参考答案
df = pd.DataFrame({'price': ['$10', '$25', '$8']})

# ❌ 没赋值! 做了个寂寞!
# df['price'].str.replace('$', '')   — 返回了新 Series 但没存
# df['price'].astype(float)          — 同上, 而且 '$10' 直接报错

# ✅ 必须赋值回去, 而且顺序不能反
df['price'] = df['price'].str.replace('$', '', regex=False)  # 先去 $
df['price'] = df['price'].astype(float)  # 再转数值

print(df)
print(f'类型: {df["price"].dtype}')  # float64

# 💡 链式写法 (一步到位):
# df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)

---
## Part 3: List ↔ Set ↔ Series 互转
---

### Q7. 去重三种写法
给定 `tags = ['python', 'sql', 'python', 'excel', 'sql', 'tableau']`，
分别用 List、Set、Series **三种方式**实现去重，输出去重后的结果。

In [40]:
tags = ['python', 'sql', 'python', 'excel', 'sql', 'tableau']
# 你的答案
set(tags)
list(dict.fromkeys(tags))
pd.Series(tags).unique()


<StringArray>
['python', 'sql', 'excel', 'tableau']
Length: 4, dtype: str

In [ ]:
# ✅ Q7 参考答案
tags = ['python', 'sql', 'python', 'excel', 'sql', 'tableau']

# 方式 1: Set (最快, 但无序)
print(f'Set: {set(tags)}')

# 方式 2: List + dict.fromkeys (保持原始顺序)
print(f'List: {list(dict.fromkeys(tags))}')

# 方式 3: Series.unique() (返回 ndarray)
print(f'Series: {pd.Series(tags).unique()}')

# 💡 笔试常问: 哪种保持顺序? → dict.fromkeys 和 Series.unique 保持顺序, set 不保持

### Q8. 集合运算实战
团队 A 掌握的技能: `{'Python', 'SQL', 'Excel', 'Tableau'}`  
团队 B 掌握的技能: `{'Python', 'R', 'SQL', 'PowerBI'}`  

请分别求出:
1. 两个团队的**所有技能** (并集)
2. **共同技能** (交集)
3. A 有但 B 没有的技能 (差集)
4. 只属于一个团队的技能 (对称差)

In [42]:
team_a = {'Python', 'SQL', 'Excel', 'Tableau'}
team_b = {'Python', 'R', 'SQL', 'PowerBI'}
# 你的答案
all = set(team_a).union(set(team_b))
union = set(team_a).intersection(set(team_b))
diff = set(team_a).difference(set(team_b))
print(f'all:{all}')
print(f'union:{union}')
print(f'diff:{diff}')




all:{'Excel', 'Python', 'R', 'Tableau', 'PowerBI', 'SQL'}
union:{'Python', 'SQL'}
diff:{'Excel', 'Tableau'}


In [43]:
# ✅ Q8 参考答案
team_a = {'Python', 'SQL', 'Excel', 'Tableau'}
team_b = {'Python', 'R', 'SQL', 'PowerBI'}

print(f'并集: {team_a | team_b}')    # 或 team_a.union(team_b)
print(f'交集: {team_a & team_b}')    # 或 team_a.intersection(team_b)
print(f'A独有: {team_a - team_b}')   # 或 team_a.difference(team_b)
print(f'对称差: {team_a ^ team_b}')  # 或 team_a.symmetric_difference(team_b)

并集: {'Excel', 'Python', 'R', 'Tableau', 'PowerBI', 'SQL'}
交集: {'Python', 'SQL'}
A独有: {'Excel', 'Tableau'}
对称差: {'Excel', 'R', 'PowerBI', 'Tableau'}


### Q9. Series vs List: 频次统计
给定 `data = ['A','B','A','C','B','A','C','C','C']`，
分别用 List 方式和 Series 方式统计每个元素出现的次数，按次数降序。

In [51]:
data = ['A','B','A','C','B','A','C','C','C']
# 你的答案
pd.Series(data).value_counts(ascending=False)
# counter(data)

C    4
A    3
B    2
Name: count, dtype: int64

In [ ]:
# ✅ Q9 参考答案
from collections import Counter
data = ['A','B','A','C','B','A','C','C','C']

# 方式 1: List — 用 Counter (Python 原生)
counter = Counter(data)
print(f'Counter: {counter.most_common()}')

# 方式 2: Series — 用 value_counts (Pandas, 自动降序)
print(f'Series:\n{pd.Series(data).value_counts()}')

# 💡 面试/笔试里如果环境有 Pandas, 优先用 value_counts, 更简洁

---
## Part 4: 缺失值处理 (NaN)
---

### Q10. fillna 的多种用法
给定以下数据，请:
1. 将 `age` 的缺失值填充为**中位数**
2. 将 `city` 的缺失值填充为 `'Unknown'`
3. 将 `score` 的缺失值用**前一个值**填充 (forward fill)

```python
df = pd.DataFrame({
    'age': [25, np.nan, 30, np.nan, 45],
    'city': ['北京', None, '上海', '广州', None],
    'score': [80, np.nan, np.nan, 90, np.nan]
})
```

In [55]:
df = pd.DataFrame({
    'age': [25, np.nan, 30, np.nan, 45],
    'city': ['北京', None, '上海', '广州', None],
    'score': [80, np.nan, np.nan, 90, np.nan]
})
# 你的答案
df['age'].median()
df['age'] = df['age'].fillna(df['age'].median())
df['city'] = df['city'].fillna('Unknown')
df['score'] = df['score'].ffill()

In [ ]:
# ✅ Q10 参考答案
df = pd.DataFrame({
    'age': [25, np.nan, 30, np.nan, 45],
    'city': ['北京', None, '上海', '广州', None],
    'score': [80, np.nan, np.nan, 90, np.nan]
})

df['age'] = df['age'].fillna(df['age'].median())     # 中位数填充: 30.0
df['city'] = df['city'].fillna('Unknown')             # 固定值填充
df['score'] = df['score'].ffill()                     # 前向填充 (80→80→80→90→90)

print(df)

# 💡 ffill = forward fill, bfill = backward fill
# 💡 数值列常用 median/mean 填充; 分类列常用固定值 'Unknown'

### Q11. NaN 拼接陷阱
给定以下数据，将 `first_name` 和 `last_name` 拼成全名。
要求: NaN 的部分不能变成字符串 `"nan"` 或 `"None"`。

```python
df = pd.DataFrame({
    'first_name': ['张', None, '李'],
    'last_name': ['三', '四', None]
})
```
期望输出: `['张三', '四', '李']`

In [ ]:
df = pd.DataFrame({
    'first_name': ['张', None, '李'],
    'last_name': ['三', '四', None]
})
# 你的答案
df['full_name'] = df[['first_name','last_name']].apply(lambda x:''.join(x.dropna().setype(str)),axis=1)

In [56]:
# ✅ Q11 参考答案
df = pd.DataFrame({
    'first_name': ['张', None, '李'],
    'last_name': ['三', '四', None]
})

# ❌ 错误: 直接 + 会变成 NaN
# df['first_name'] + df['last_name']  → ['张三', NaN, NaN]

# ❌ 错误: fillna('') 后 + 会有多余空位
# (df['first_name'].fillna('') + df['last_name'].fillna(''))  → 能用但不够优雅

# ✅ 推荐: apply + join + dropna (你在 38 号 notebook 学过的!)
df['full_name'] = df[['first_name', 'last_name']].apply(
    lambda x: ''.join(x.dropna().astype(str)), axis=1
)
print(df)

# 💡 这是 38 号 notebook Level 3 的进阶一行流写法

  first_name last_name full_name
0          张         三        张三
1        NaN         四         四
2          李       NaN         李


---
## Part 5: apply vs 向量化
---

### Q12. 判断分箱: apply vs np.where vs pd.cut
给定 `ages = pd.Series([15, 25, 35, 45, 55, 65])`,
将年龄分为:
- `<18`: '未成年'
- `18-40`: '青年'
- `40-60`: '中年'
- `>60`: '老年'

请用**两种方式**实现 (apply 和 np.select)

In [59]:
ages = pd.Series([15, 25, 35, 45, 55, 65])
# 你的答案
ages = pd.cut(ages,bins=[0,18,40,60,100],labels=['未成年','青年','中年','老年'])
ages

0    未成年
1     青年
2     青年
3     中年
4     中年
5     老年
dtype: category
Categories (4, str): ['未成年' < '青年' < '中年' < '老年']

In [ ]:
# ✅ Q12 参考答案
ages = pd.Series([15, 25, 35, 45, 55, 65])

# 方法 1: apply + lambda (慢, 但直觉)
def classify(age):
    if age < 18: return '未成年'
    elif age <= 40: return '青年'
    elif age <= 60: return '中年'
    else: return '老年'

print('apply:', ages.apply(classify).tolist())

# 方法 2: np.select (快, 向量化)
conditions = [ages < 18, ages <= 40, ages <= 60, ages > 60]
labels = ['未成年', '青年', '中年', '老年']
print('np.select:', list(np.select(conditions, labels, default='未知')))

# 方法 3: pd.cut (最简洁)
print('pd.cut:', pd.cut(ages, bins=[0,18,40,60,100], labels=['未成年','青年','中年','老年']).tolist())

# 💡 2个分箱用 np.where, 多分箱用 np.select 或 pd.cut

### Q13. .str 访问器 vs apply
给定以下邮箱列表，提取 `@` 前面的用户名部分。请用**两种方式**:
```python
emails = pd.Series(['alice@gmail.com', 'bob@qq.com', None, 'charlie@163.com'])
```

In [ ]:
emails = pd.Series(['alice@gmail.com', 'bob@qq.com', None, 'charlie@163.com'])
# 你的答案


In [ ]:
# ✅ Q13 参考答案
emails = pd.Series(['alice@gmail.com', 'bob@qq.com', None, 'charlie@163.com'])

# 方法 1: .str 访问器 (推荐! 自动处理 NaN)
print(emails.str.split('@').str[0])

# 方法 2: apply (需要手动处理 NaN)
print(emails.apply(lambda x: x.split('@')[0] if pd.notna(x) else x))

# 💡 .str 访问器: 自动跳过 NaN, 代码更短
# 💡 apply: 需要手动 if pd.notna(x), 否则 None.split() 报错

### Q14. 布尔掩码 vs apply
给定以下数据，新增一列 `is_vip`：消费金额 > 1000 **且** 购买次数 >= 5 的为 VIP (1)，否则为 0。
```python
df = pd.DataFrame({
    'amount': [500, 1200, 3000, 800, 1500],
    'purchases': [3, 6, 2, 10, 5]
})
```
**不允许使用 apply!**

In [61]:
df = pd.DataFrame({
    'amount': [500, 1200, 3000, 800, 1500],
    'purchases': [3, 6, 2, 10, 5]
})
# 你的答案 (不许用 apply!)
df['is_vip'] = np.where((df['amount']>1000) & (df['purchases']>=5),1,0)
df['is_vip']

0    0
1    1
2    0
3    0
4    1
Name: is_vip, dtype: int64

In [ ]:
# ✅ Q14 参考答案
df = pd.DataFrame({
    'amount': [500, 1200, 3000, 800, 1500],
    'purchases': [3, 6, 2, 10, 5]
})

# 向量化布尔运算 — 比 apply 快 100 倍
df['is_vip'] = ((df['amount'] > 1000) & (df['purchases'] >= 5)).astype(int)
print(df)

# 💡 布尔条件用 & (与), | (或), ~ (非)
# 💡 每个条件必须加括号! (df['a'] > 1) & (df['b'] < 2)

### Q15. 综合题: 数据清洗全流程
给定以下脏数据，完成清洗:
```python
df = pd.DataFrame({
    'name': [' Alice ', 'BOB', ' charlie', 'Alice ', None],
    'salary': ['$5,000', '$3,200', '$8,100', '$5,000', '$4,500'],
    'dept': ['IT', 'hr', 'IT', 'it', None]
})
```
清洗要求:
1. `name`: 去首尾空格 + 统一首字母大写
2. `salary`: 去掉 `$` 和 `,`，转为 float
3. `dept`: 统一大写
4. `dept` 缺失值填充为 `'Unknown'`
5. 按 `name` 去重 (保留第一条)

In [63]:
df = pd.DataFrame({
    'name': [' Alice ', 'BOB', ' charlie', 'Alice ', None],
    'salary': ['$5,000', '$3,200', '$8,100', '$5,000', '$4,500'],
    'dept': ['IT', 'hr', 'IT', 'it', None]
})
# 你的答案
df['name'] = df['name'].str.strip()
df['salary'] = df['salary'].replace(r'[$,]','',regex=True).astype(float)
df['dept'] = df['dept'].str.upper().fillna('Unknown')
df.drop_duplicates(subset='name',keep='first')

,name,salary,dept
0,Alice,5000.0,IT
1,BOB,3200.0,HR
2,charlie,8100.0,IT
4,NaN,4500.0,Unknown


In [64]:
# ✅ Q15 参考答案
df = pd.DataFrame({
    'name': [' Alice ', 'BOB', ' charlie', 'Alice ', None],
    'salary': ['$5,000', '$3,200', '$8,100', '$5,000', '$4,500'],
    'dept': ['IT', 'hr', 'IT', 'it', None]
})

# 1. name: 去空格 + 首字母大写
df['name'] = df['name'].str.strip().str.title()

# 2. salary: 去 $ 和 , → 转 float
df['salary'] = df['salary'].str.replace('[$,]', '', regex=True).astype(float)

# 3. dept: 统一大写
df['dept'] = df['dept'].str.upper()

# 4. dept 填充缺失值
df['dept'] = df['dept'].fillna('UNKNOWN')

# 5. 按 name 去重
df = df.drop_duplicates(subset='name', keep='first')

print(df)

# 💡 .str 链式操作: strip() → title/upper/lower → replace
# 💡 去重前一定要先做标准化 (大小写/空格), 否则 'Alice' 和 ' Alice ' 被视为不同

      name  salary     dept
0    Alice  5000.0       IT
1      Bob  3200.0       HR
2  Charlie  8100.0       IT
4      NaN  4500.0  UNKNOWN


---
## 📋 完成后自检

- [ ] Part 1: 能区分 `re.sub` / `str.replace` / `.str.replace` 三者
- [ ] Part 2: 知道 `value_counts` 会覆盖之前的排序
- [ ] Part 3: 能熟练做 List ↔ Set ↔ Series 互转和去重
- [ ] Part 4: 知道 `fillna` 的多种用法, 拼接时能正确处理 NaN
- [ ] Part 5: 能用向量化方式替代 apply (布尔掩码, np.select, .str 访问器)